# `Five Card Stud` 项目练习

**学习目标：** 本项目按核心数据结构、牌型判断和项目作业组织，目标是用面向对象方式完成纸牌游戏的主要建模过程。

## 核心数据结构

本项目用类表示纸牌、牌堆、手牌、筹码和玩家，并在此基础上实现牌型判断。


In [ ]:
import random
from collections import Counter
from functools import total_ordering


### 一张牌

一张牌包含花色 `suit`、序数 `rank`、点数 `point` 和比较值 `value`。


In [ ]:
class Card:
    suits = ["spade", "heart", "club", "diamond"]
    ranks = ["2", "3", "4", "5", "6", "7", "8", "9", "10", "J", "Q", "K", "A"]
    suit_codes = {"spade": "♠", "heart": "♡", "club": "♣", "diamond": "♢"}
    suit_values = {"spade": 4, "heart": 3, "club": 2, "diamond": 1}
    rank_points = {"2": 2, "3": 3, "4": 4, "5": 5, "6": 6, "7": 7, "8": 8, "9": 9, "10": 10, "J": 10, "Q": 10, "K": 10, "A": 11}
    rank_values = {"2": 2, "3": 3, "4": 4, "5": 5, "6": 6, "7": 7, "8": 8, "9": 9, "10": 10, "J": 11, "Q": 12, "K": 13, "A": 14}

    def __init__(self, suit="spade", rank="A"):
        self.suit = suit
        self.rank = rank
        self.point = self.rank_points[rank]
        self.value = self.rank_values[rank]
        self.suit_value = self.suit_values[suit]

    def __str__(self):
        return f"{self.suit_codes[self.suit]}{self.rank}"


### 一副牌

`Deck` 负责创建、洗牌和发牌。默认使用完整的 `52` 张牌。


In [ ]:
class Deck:
    def __init__(self, shuffle=True):
        self.cards = [Card(suit, rank) for suit in Card.suits for rank in Card.ranks]
        if shuffle:
            self.shuffle()

    def shuffle(self):
        random.shuffle(self.cards)

    def deal(self):
        return self.cards.pop()

    def __str__(self):
        return " ".join(str(card) for card in self.cards)


### 一手牌

`Hand` 保存玩家当前手牌，并提供添加牌、展示暗牌和比较点数的能力。


In [ ]:
@total_ordering
class Hand:
    def __init__(self):
        self.cards = []
        self.points = 0
        self.aces = 0

    def add_card(self, card):
        self.cards.append(card)
        self.points += card.point
        if card.rank == "A":
            self.aces += 1
        self._adjust_ace_point()

    def draw_random_cards(self, count=5):
        deck = Deck()
        self.cards = []
        self.points = 0
        self.aces = 0
        for _ in range(count):
            self.add_card(deck.deal())

    def _adjust_ace_point(self):
        while self.points > 21 and self.aces:
            self.points -= 10
            self.aces -= 1

    def show(self, show_all=True):
        cards = [str(card) for card in self.cards]
        if not show_all and cards:
            cards[0] = "<Hole Card>"
        return " ".join(cards)

    def __str__(self):
        return self.show(show_all=True)

    def __eq__(self, other):
        return self.points == other.points

    def __lt__(self, other):
        return self.points < other.points


### 筹码与玩家

`Chips` 管理总筹码和本轮下注，`Player` 管理玩家手牌、下注和拿牌动作。


In [ ]:
class Chips:
    def __init__(self, total=100, ante=0):
        self.total = total
        self.ante = ante

    def win_bet(self, amount=0):
        self.total += amount

    def lose_bet(self):
        self.total -= self.ante


class Player:
    def __init__(self, name="player", hand=None, chips=None):
        self.name = name
        self.hand = hand if hand is not None else Hand()
        self.chips = chips if chips is not None else Chips()

    def init_cards(self, deck, count=2):
        self.hand = Hand()
        for _ in range(count):
            self.hand.add_card(deck.deal())

    def can_bet(self, last_bet):
        return self.chips.total - self.chips.ante >= last_bet

    def bet(self, ante):
        if ante < 0:
            raise ValueError("ante must be non-negative")
        self.chips.ante += ante

    def bet_all(self):
        self.chips.ante = self.chips.total

    def hit(self, deck):
        self.hand.add_card(deck.deal())

    def show_public_cards(self):
        return self.hand.show(show_all=False)


## 牌型判断

重点问题：

1. 如何判断一手牌的牌型？
2. 如何比较两手牌的大小？


In [ ]:
HAND_RANKS = {
    9: "straight flush",
    8: "four of a kind",
    7: "full house",
    6: "flush",
    5: "straight",
    4: "three of a kind",
    3: "two pair",
    2: "one pair",
    1: "high card",
}


def is_straight(rank_values):
    unique_values = sorted(set(rank_values))
    return len(unique_values) == 5 and (unique_values[-1] - unique_values[0] == 4 or unique_values == [2, 3, 4, 5, 14])


def hand_pattern(hand):
    suits = [card.suit for card in hand.cards]
    ranks = [card.value for card in hand.cards]
    suit_counts = sorted(Counter(suits).values())
    rank_counts = sorted(Counter(ranks).values())

    if suit_counts == [5] and is_straight(ranks):
        return 9
    if rank_counts == [1, 4]:
        return 8
    if rank_counts == [2, 3]:
        return 7
    if suit_counts == [5]:
        return 6
    if is_straight(ranks):
        return 5
    if rank_counts == [1, 1, 3]:
        return 4
    if rank_counts == [1, 2, 2]:
        return 3
    if rank_counts == [1, 1, 1, 2]:
        return 2
    return 1


### 示例：随机生成一手牌并判断牌型


In [ ]:
hand = Hand()
hand.draw_random_cards(5)
pattern = hand_pattern(hand)
print(hand)
print(HAND_RANKS[pattern])


## 项目作业

在上述类的基础上，补全 `Five Card Stud` 的完整游戏流程。
